# Data Pipelines with tf.data: map, batch, prefetch, cache, and shuffle Best Practices

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/tensorflow_4)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q tensorflow mlflow

In [ ]:
import tensorflow as tf
import numpy as np

# In-memory dataset
X = np.random.randn(1000, 28, 28, 1).astype(np.float32)
y = np.random.randint(0, 10, 1000).astype(np.int32)

dataset = tf.data.Dataset.from_tensor_slices((X, y))
print(f"Dataset element spec: {dataset.element_spec}")
print(f"Dataset cardinality: {dataset.cardinality().numpy()} elements")

In [ ]:
import tensorflow as tf
import numpy as np

dataset = tf.data.Dataset.range(10)

# Small buffer: only shuffles within a 3-element window
small_shuffle = dataset.shuffle(buffer_size=3, seed=42)
print("Small buffer:", list(small_shuffle.as_numpy_iterator()))

# Full buffer: truly random
full_shuffle = dataset.shuffle(buffer_size=10, seed=42)
print("Full buffer: ", list(full_shuffle.as_numpy_iterator()))

In [ ]:
import tensorflow as tf
import numpy as np

dataset = tf.data.Dataset.from_tensor_slices(
    np.random.randint(0, 256, (100, 32, 32, 3), dtype=np.uint8)
)

def preprocess(image):
    # Normalize to [0, 1]
    image = tf.cast(image, tf.float32) / 255.0
    # Random horizontal flip (training augmentation)
    image = tf.image.random_flip_left_right(image)
    return image

processed = dataset.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
sample = next(iter(processed))
print(f"Output dtype: {sample.dtype}")
print(f"Output range: [{sample.numpy().min():.3f}, {sample.numpy().max():.3f}]")
print(f"Output shape: {sample.shape}")

In [ ]:
import tensorflow as tf
import time
import numpy as np

def slow_preprocess(x):
    """Simulates slow preprocessing (e.g., image decoding from disk)."""
    tf.py_function(lambda: time.sleep(0.001), [], [])
    return tf.cast(x, tf.float32) / 255.0

dataset = tf.data.Dataset.from_tensor_slices(
    np.random.randint(0, 256, (200, 10), dtype=np.uint8)
)

# Without cache: slow preprocessing runs every epoch
ds_no_cache = dataset.map(slow_preprocess).batch(32)

# With cache: preprocessing runs once, cached results reused
ds_cached = dataset.map(slow_preprocess).cache().batch(32)

for ds_name, ds in [("no cache", ds_no_cache), ("cached", ds_cached)]:
    times = []
    for epoch in range(3):
        start = time.time()
        for _ in ds:
            pass
        times.append(time.time() - start)
    print(f"{ds_name}: {[f'{t:.2f}s' for t in times]}")

In [ ]:
import tensorflow as tf
import numpy as np

dataset = tf.data.Dataset.from_tensor_slices(
    np.arange(10, dtype=np.float32)
)

batched = dataset.batch(3, drop_remainder=False)
for batch in batched:
    print(batch.numpy())

In [ ]:
import tensorflow as tf
import numpy as np

dataset = tf.data.Dataset.from_tensor_slices(
    (np.random.randn(1000, 224, 224, 3).astype(np.float32),
     np.random.randint(0, 1000, 1000))
)

# Without prefetch: CPU and GPU work sequentially
ds_no_prefetch = dataset.batch(32)

# With prefetch: CPU prepares N+1 while GPU trains on N
ds_prefetch = dataset.batch(32).prefetch(tf.data.AUTOTUNE)

print(f"No prefetch: element_spec = {ds_no_prefetch.element_spec}")
print(f"Prefetch:    element_spec = {ds_prefetch.element_spec}")
print("\nprefetch(AUTOTUNE) lets TF determine the optimal buffer size automatically.")

In [ ]:
import tensorflow as tf
import numpy as np

def augment(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.1)
    return image, label

def build_pipeline(X, y, batch_size=32, training=True):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))

    if training:
        dataset = dataset.shuffle(buffer_size=len(X), reshuffle_each_iteration=True)
        dataset = dataset.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        dataset = dataset.map(
            lambda x, y: (tf.cast(x, tf.float32) / 255.0, y),
            num_parallel_calls=tf.data.AUTOTUNE
        )

    dataset = dataset.cache()
    dataset = dataset.batch(batch_size, drop_remainder=training)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

np.random.seed(42)
X_train = np.random.randint(0, 256, (1000, 32, 32, 3), dtype=np.uint8)
y_train = np.random.randint(0, 10, 1000, dtype=np.int32)
X_val   = np.random.randint(0, 256, (200,  32, 32, 3), dtype=np.uint8)
y_val   = np.random.randint(0, 10, 200, dtype=np.int32)

train_ds = build_pipeline(X_train, y_train, batch_size=32, training=True)
val_ds   = build_pipeline(X_val,   y_val,   batch_size=32, training=False)

# Verify shapes
for x_batch, y_batch in train_ds.take(1):
    print(f"Train batch: x={x_batch.shape}, y={y_batch.shape}, x.dtype={x_batch.dtype}")

for x_batch, y_batch in val_ds.take(1):
    print(f"Val batch:   x={x_batch.shape}, y={y_batch.shape}, x.dtype={x_batch.dtype}")

In [ ]:
import tensorflow as tf
import numpy as np

data = tf.data.Dataset.from_tensor_slices(np.ones((100, 28, 28), dtype=np.float32))

# WRONG: batching before map means map receives (batch_size, 28, 28) tensors
# Your per-sample function must handle batches, not samples
wrong = data.batch(16).map(lambda x: x / 255.0)

# CORRECT: map first (per-sample), then batch
correct = data.map(lambda x: x / 255.0).batch(16)

print(f"Wrong:   {wrong.element_spec}")
print(f"Correct: {correct.element_spec}")

In [ ]:
import tensorflow as tf
import numpy as np

data = tf.data.Dataset.from_tensor_slices(np.ones(100, dtype=np.float32))

# WRONG: caches batches — shuffle happens before cache so re-shuffling
# each epoch requires re-reading from cache (loses shuffle freshness)
wrong_order = data.shuffle(100).batch(16).cache().prefetch(tf.data.AUTOTUNE)

# CORRECT: cache pre-batched, shuffled data; batch and prefetch after
correct_order = data.shuffle(100).cache().batch(16).prefetch(tf.data.AUTOTUNE)

print("WRONG:   shuffle → batch → cache → prefetch")
print("CORRECT: shuffle → cache → batch → prefetch")
print("\nWith correct order, each epoch reshuffles the cached per-sample data.")

In [ ]:
import tensorflow as tf
import numpy as np
import time

def benchmark_pipeline(ds, steps=50):
    start = time.perf_counter()
    for i, _ in enumerate(ds):
        if i >= steps:
            break
    return (time.perf_counter() - start) / steps

X = np.random.randn(2000, 128, 128, 3).astype(np.float32)
y = np.random.randint(0, 100, 2000).astype(np.int32)

base_ds = tf.data.Dataset.from_tensor_slices((X, y))

configs = {
    "no optimization":  base_ds.batch(32),
    "+ map parallel":   base_ds.map(lambda x, y: (x/255.0, y), num_parallel_calls=tf.data.AUTOTUNE).batch(32),
    "+ cache":          base_ds.map(lambda x, y: (x/255.0, y), num_parallel_calls=tf.data.AUTOTUNE).cache().batch(32),
    "+ prefetch":       base_ds.map(lambda x, y: (x/255.0, y), num_parallel_calls=tf.data.AUTOTUNE).cache().batch(32).prefetch(tf.data.AUTOTUNE),
}

for name, ds in configs.items():
    # Warm up
    for _ in ds.take(1): pass
    avg_time = benchmark_pipeline(ds)
    print(f"{name:25s}: {avg_time*1000:.1f} ms/batch")